# Customer x Product Bridge EDA

Objective: Connect customer behavior segments with product/category preferences.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

USE_SAMPLE = True
TOP_DEPARTMENTS = 8

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists() and (parent / "data").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from instacart_quality import load_eda_base

tables = load_eda_base(use_sample=USE_SAMPLE, project_root=PROJECT_ROOT)
orders = tables["orders"]
order_products = tables["order_products_prior"]
products = tables["products"]
departments = tables["departments"]

df = order_products.merge(orders, on="order_id", how="left").merge(
    products[["product_id", "department_id"]], on="product_id", how="left"
).merge(departments, on="department_id", how="left")
df.head()

In [ ]:
customer_profile = (
    df.groupby("user_id")
    .agg(
        reorder_rate=("reordered", "mean"),
        avg_basket_pos=("add_to_cart_order", "mean"),
        total_lines=("product_id", "count"),
    )
    .reset_index()
)
customer_profile["segment"] = pd.qcut(
    customer_profile["reorder_rate"],
    q=3,
    labels=["Low Reorder", "Mid Reorder", "High Reorder"],
)

bridge = df.merge(customer_profile[["user_id", "segment"]], on="user_id", how="left")
heat = (
    bridge.groupby(["segment", "department"]).size().rename("order_lines").reset_index()
)
top_departments = (
    heat.groupby("department")["order_lines"].sum().sort_values(ascending=False).head(TOP_DEPARTMENTS).index
)
heat = heat[heat["department"].isin(top_departments)]
pivot = heat.pivot(index="segment", columns="department", values="order_lines").fillna(0)

plt.figure(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues")
plt.title("Customer Segment vs Department Demand")
plt.tight_layout()
plt.show()